## Unitree G1

### Setup

In [1]:
import gtdynamics as gtd
import gtsam 
import numpy as np
import plotly 
import mujoco
import mujoco.viewer
import mediapy as media

#from gtsam import Point3

In [2]:
H1_PATH = "../../models/h1_description"
URDF_PATH = H1_PATH + "/urdf/h1.urdf"
MJCF_PATH = H1_PATH + "/mjcf/scene.xml"

In [3]:
robot = gtd.CreateRobotFromFile(URDF_PATH)

In [4]:
model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

In [ ]:
mujoco.viewer.launch(model, data)

In [5]:
# Noise models.
sigma_dynamics = 1e-5    # std of dynamics constraints.
sigma_objectives = 1e-6  # std of additional objectives.
sigma_joints = 1.85e-4   # 1.85e-4

dynamics_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_dynamics)
dynamics_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_dynamics)
dynamics_model_1_2 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_joints)
objectives_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_objectives)
objectives_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_objectives)


In [21]:
link_names = [(link.id(), link.name()) for link in robot.links()]
link_names.sort()
print(link_names)

[(0, 'pelvis'), (1, 'left_hip_yaw_link'), (2, 'left_hip_roll_link'), (3, 'left_hip_pitch_link'), (4, 'left_knee_link'), (5, 'left_ankle_link'), (6, 'right_hip_yaw_link'), (7, 'right_hip_roll_link'), (8, 'right_hip_pitch_link'), (9, 'right_knee_link'), (10, 'right_ankle_link'), (11, 'torso_link'), (12, 'left_shoulder_pitch_link'), (13, 'left_shoulder_roll_link'), (14, 'left_shoulder_yaw_link'), (15, 'left_elbow_link'), (16, 'right_shoulder_pitch_link'), (17, 'right_shoulder_roll_link'), (18, 'right_shoulder_yaw_link'), (19, 'right_elbow_link')]


In [8]:
joint_names = [(joint.id(), joint.name()) for joint in robot.joints()]
joint_names.sort()
print(joint_names)

[(0, 'left_hip_yaw_joint'), (1, 'left_hip_roll_joint'), (2, 'left_hip_pitch_joint'), (3, 'left_knee_joint'), (4, 'left_ankle_joint'), (5, 'right_hip_yaw_joint'), (6, 'right_hip_roll_joint'), (7, 'right_hip_pitch_joint'), (8, 'right_knee_joint'), (9, 'right_ankle_joint'), (10, 'torso_joint'), (11, 'left_shoulder_pitch_joint'), (12, 'left_shoulder_roll_joint'), (13, 'left_shoulder_yaw_joint'), (14, 'left_elbow_joint'), (15, 'right_shoulder_pitch_joint'), (16, 'right_shoulder_roll_joint'), (17, 'right_shoulder_yaw_joint'), (18, 'right_elbow_joint')]


In [9]:
left_ankle = robot.link("left_ankle_link")
right_ankle = robot.link("right_ankle_link")
contact_in_com = gtsam.Point3(np.array([-0.048, 0, 0.045]))
point_on_left_ankle = gtd.PointOnLink(left_ankle, contact_in_com)
point_on_right_ankle = gtd.PointOnLink(right_ankle, contact_in_com)

In [10]:
time_horizon = 5.0  # seconds
num_steps = 100
dt = time_horizon / num_steps

### Building and Optimizing FG

In [65]:
print("Creating dynamics graph with gravity and contact physics...")
gravity_vec = np.array([0, 0, -9.8])
opt = gtd.OptimizerSetting(sigma_dynamics)
graph_builder = gtd.DynamicsGraph(opt, gravity_vec, None)
graph = graph_builder.trajectoryFG(robot, num_steps, dt)

Creating dynamics graph with gravity and contact physics...


In [66]:
# Add initial conditions on angles and velocities of each joint
keyframe_values = [0, 0, -0.4, 0.8, -0.4,
    0, 0, -0.4, 0.8, -0.4,
    0,
    0, 0, 0, 0,
    0, 0, 0, 0]

idx = 0
for id, name in joint_names:
    graph.addPriorDouble(gtd.JointAngleKey(id, 0), keyframe_values[idx], dynamics_model_1)
    graph.addPriorDouble(gtd.JointVelKey(id, 0), 0, dynamics_model_1)
    idx += 1

In [ ]:
# CONTACT CONSTRAINTS: Keep both feet on the ground
print("Adding ground contact constraints for both ankles...")

# Contact points on feet (where they touch ground)
contact_point_offset = np.array([0, 0, -0.045])  # Contact point below ankle
gravity_vector = np.array([0, 0, -9.8])

# Height constraint noise model
contact_noise = gtsam.noiseModel.Isotropic.Sigma(1, 1e-4)  # Height constraint

for t in range(num_steps):
    # === LEFT FOOT CONTACT ===
    left_pose_key = gtd.PoseKey(left_ankle.id(), t)
    # Height constraint (foot must touch ground)
    left_height_factor = gtd.ContactHeightFactor(left_pose_key, contact_noise, 
                                                 contact_point_offset, gravity_vector, 0.0)
    graph.add(left_height_factor)
    
    # === RIGHT FOOT CONTACT ===
    right_pose_key = gtd.PoseKey(right_ankle.id(), t)
    # Height constraint (foot must touch ground)
    right_height_factor = gtd.ContactHeightFactor(right_pose_key, contact_noise,
                                                  contact_point_offset, gravity_vector, 0.0)
    graph.add(right_height_factor)
    
# Add prior on the pelvis
pelvis_link = robot.link("pelvis")
point_com = gtsam.Point3(np.array([0.0, 0.0, 0.0]))  # Point on pelvis (COM)

# Target pelvis position with slight forward lean to challenge balance
keyframe_pelvis_position = gtsam.Point3(np.array([0.02, 0.0, 0.98]))  # 2cm forward lean + keyframe height
balance_noise = gtsam.noiseModel.Isotropic.Sigma(3, 0.05)  # Moderate constraint

# Apply balance target at a few key timesteps
balance_timesteps = [20, 40, 60, 80]  # Mid-trajectory balance points
for t in balance_timesteps:
    pose_key = gtd.PoseKey(pelvis_link.id(), t)
    factor = gtd.PointGoalFactor(pose_key, balance_noise, point_com, keyframe_pelvis_position)
    graph.add(factor)

Adding ground contact constraints for both ankles...
Added 200 contact height constraints:
  - Both ankles constrained to ground level (z = 0)
  - trajectoryFG() already includes friction and dynamics constraints!
  - Contact physics will now require torques to maintain balance!
Adding balance challenge...
Added 4 balance challenge constraints
  - Target: 2cm forward lean from keyframe position
  - This will force the robot to use ankle/hip torques for balance!
Added final stability constraint (return to keyframe position)


In [ ]:
# Add minimum torque factors for all joints to encourage energy-efficient motion
torque_noise = gtsam.noiseModel.Isotropic.Sigma(1, 2.0)

for t in range(num_steps):
    for joint_id, joint_name in joint_names:
        torque_key = gtd.TorqueKey(joint_id, t)
        min_torque_factor = gtd.MinTorqueFactor(torque_key, torque_noise)
        graph.add(min_torque_factor)

In [68]:
# Print size of a graph, inspect graph using GTDKeyFormatter
print(f"Factor graph size: {graph.size()} factors")

# Print detailed information about the graph structure
print("\nFactor graph contents:")
# Print just the first few factors to get a sample
print(f"\nFirst 5 factors in the graph:")
for i in range(min(5, graph.size())):
    factor = graph.at(i)
    print(f"Factor {i}: {type(factor).__name__}")
    factor.print(f"  Factor {i}", gtd.GTDKeyFormatter)

Factor graph size: 15658 factors

Factor graph contents:

First 5 factors in the graph:
Factor 0: NonlinearFactor
  Factor 0  keys = { p[4]0 p[5]0 q(4)0 }
isotropic dim=6 sigma=1e-05
ExpressionFactor with measurement: [
	0;
	0;
	0;
	0;
	0;
	0
]
Factor 1: NonlinearFactor
  Factor 1  keys = { p[14]0 p[15]0 q(14)0 }
isotropic dim=6 sigma=1e-05
ExpressionFactor with measurement: [
	0;
	0;
	0;
	0;
	0;
	0
]
Factor 2: NonlinearFactor
  Factor 2  keys = { p[2]0 p[3]0 q(2)0 }
isotropic dim=6 sigma=1e-05
ExpressionFactor with measurement: [
	0;
	0;
	0;
	0;
	0;
	0
]
Factor 3: NonlinearFactor
  Factor 3  keys = { p[1]0 p[2]0 q(1)0 }
isotropic dim=6 sigma=1e-05
ExpressionFactor with measurement: [
	0;
	0;
	0;
	0;
	0;
	0
]
Factor 4: NonlinearFactor
  Factor 4  keys = { p[0]0 p[1]0 q(0)0 }
isotropic dim=6 sigma=1e-05
ExpressionFactor with measurement: [
	0;
	0;
	0;
	0;
	0;
	0
]


In [ ]:
# Create initial guess with custom joint angle values
print("Creating custom initial guess...")

# Start with the default zero trajectory
initializer = gtd.Initializer()
initial_values = initializer.ZeroValuesTrajectory(robot, num_steps, 0, 0.0, None)

print(f"Initial values size: {initial_values.size()}")


# Set joint angles for ALL timesteps to keyframe values (creates a "static" initial guess)
print("\nSetting joint angles for all timesteps to keyframe values...")
for t in range(num_steps):
    idx = 0
    for joint_id, joint_name in joint_names:
        angle_key = gtd.JointAngleKey(joint_id, t)
        if initial_values.exists(angle_key):
            initial_values.update(angle_key, keyframe_values[idx])
        idx += 1

print("Initial guess created!")
print("  - Joint angles: Set to keyframe values for all timesteps")
print("  - Joint velocities: All zeros")
print("  - Joint accelerations: All zeros") 
print("  - Torques: All zeros")

Creating custom initial guess...
Initial values size: 17574

Setting joint angles for all timesteps to keyframe values...
Initial guess created!
  - Joint angles: Set to keyframe values for all timesteps
  - Joint velocities: All zeros
  - Joint accelerations: All zeros
  - Torques: All zeros
  - Link poses: Computed from forward kinematics


In [70]:
# Optimize with summary output
print("Setting up optimizer...")

# Configure optimizer parameters
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY")  # Shows iteration summary
params.setMaxIterations(100)      # Maximum iterations
params.setRelativeErrorTol(1e-5)  # Convergence tolerance
params.setAbsoluteErrorTol(1e-5)  # Absolute error tolerance

print(f"Optimizer settings:")
print(f"  - Verbosity: SUMMARY")
print(f"  - Max iterations: {params.getMaxIterations()}")
print(f"  - Relative error tolerance: {params.getRelativeErrorTol()}")
print(f"  - Absolute error tolerance: {params.getAbsoluteErrorTol()}")

# Create optimizer
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_values, params)

print(f"\nStarting optimization...")
print(f"  - Problem size: {graph.size()} factors, {initial_values.size()} variables")
print(f"  - Initial error: {graph.error(initial_values):.6e}")


Setting up optimizer...
Optimizer settings:
  - Verbosity: SUMMARY
  - Max iterations: 100
  - Relative error tolerance: 1e-05
  - Absolute error tolerance: 1e-05

Starting optimization...
  - Problem size: 15658 factors, 17574 variables
  - Initial error: 1.999431e+16


In [71]:
# Run optimization
result = optimizer.optimize()

Initial error: 2e+16, values: 17574
iter      cost      cost_change    lambda  success iter_time
   0          inf            0      1e-05      0        0.2
iter      cost      cost_change    lambda  success iter_time
   0        5e+13        2e+16     0.0001      1       0.27
   1          inf            0      1e-05      0       0.11
   1          inf            0     0.0001      0       0.19
   1        2e+12      4.8e+13      0.001      1       0.26
   2          inf            0     0.0001      0       0.15
   2      2.7e+10      1.9e+12      0.001      1       0.27
   3          inf            0     0.0001      0       0.21
   3      3.8e+07      2.7e+10      0.001      1       0.26
   4      1.1e+03      3.8e+07     0.0001      1       0.26
   5          inf            0      1e-05      0        0.1
   5       0.0013      1.1e+03     0.0001      1       0.25
   6          inf            0      1e-05      0       0.11
   6      4.2e-10       0.0013     0.0001      1       0.26
  

In [ ]:
result

### Simulation in Mujoco

In [73]:
print("Setting up MuJoCo simulation...")

# Reset MuJoCo to initial state
mujoco.mj_resetData(model, data)

# Use the predefined "home" keyframe from h1.xml
print("Loading predefined 'home' keyframe from h1.xml...")

# Find the keyframe by name
keyframe_id = -1
for i in range(model.nkey):
    key_name = model.key(i).name
    if key_name == "home":
        keyframe_id = i
        break

if keyframe_id >= 0:
    # Reset to the keyframe pose
    mujoco.mj_resetDataKeyframe(model, data, keyframe_id)
    print(f"Successfully loaded keyframe '{model.key(keyframe_id).name}'")
    
    # Print the keyframe joint positions for verification
    print("Keyframe joint positions:")
    print(f"  Pelvis position: {data.qpos[0:3]}")
    print(f"  Pelvis orientation (quat): {data.qpos[3:7]}")
    print(f"  Joint angles: {data.qpos[7:]}")
    
else:
    print("Warning: 'home' keyframe not found, using default pose")
    # Fallback to manual setup if keyframe not found
    data.qpos[0:3] = [0.0, 0.0, 0.98]  # From keyframe: slightly lower than 1.1
    data.qpos[3:7] = [1.0, 0.0, 0.0, 0.0]  # Identity quaternion

# Forward kinematics to update the state
mujoco.mj_forward(model, data)

print(f"Starting pelvis position: {data.qpos[0:3]}")
print("Starting simulation with optimized torques...")

# Create actuator mapping (joint name to actuator index)
actuator_map = {}
for i in range(model.nu):
    actuator_name = model.actuator(i).name
    if actuator_name:
        actuator_map[actuator_name] = i

print(f"Found {len(actuator_map)} actuators")

# Simulate and render
frames = []
with mujoco.Renderer(model, height=480, width=640) as renderer:
    for t in range(num_steps):
        # Apply optimized torques for this timestep
        for joint_id, joint_name in joint_names:
            # Get the optimized torque for this joint at this timestep
            torque_key = gtd.TorqueKey(joint_id, t)
            if result.exists(torque_key):
                torque = result.atDouble(torque_key)
                print(f"Applying torque {torque}")
                
                # Apply torque to corresponding actuator
                if joint_name in actuator_map:
                    actuator_idx = actuator_map[joint_name]
                    data.ctrl[actuator_idx] = torque
        
        # Step simulation forward for dt seconds
        target_time = data.time + dt
        while data.time < target_time:
            mujoco.mj_step(model, data)
        
        # Render and capture frame
        renderer.update_scene(data)
        frames.append(renderer.render())
        
        # Print progress every 20 steps
        if t % 20 == 0:
            pelvis_pos = data.qpos[0:3]
            print(f"  Step {t:3d}/{num_steps}: time={data.time:.2f}s, pelvis_pos=[{pelvis_pos[0]:.3f}, {pelvis_pos[1]:.3f}, {pelvis_pos[2]:.3f}]")

print(f"Simulation complete! Generated {len(frames)} frames")

# Display video
if frames:
    print("Displaying optimized trajectory...")
    media.show_video(frames, fps=1/dt)
else:
    print("No frames generated - check simulation setup")

Setting up MuJoCo simulation...
Loading predefined 'home' keyframe from h1.xml...
Successfully loaded keyframe 'home'
Keyframe joint positions:
  Pelvis position: [0.   0.   0.98]
  Pelvis orientation (quat): [1. 0. 0. 0.]
  Joint angles: [ 0.   0.  -0.4  0.8 -0.4  0.   0.  -0.4  0.8 -0.4  0.   0.   0.   0.
  0.   0.   0.   0.   0. ]
Starting pelvis position: [0.   0.   0.98]
Starting simulation with optimized torques...
Found 19 actuators
Applying torque 0.018923093298626303
Applying torque -0.0006185107674193246
Applying torque -0.033508037802375094
Applying torque -0.09093108880459924
Applying torque -0.000457598888332444
Applying torque -0.017988640948198062
Applying torque 0.014634241657902487
Applying torque -0.023406584129706726
Applying torque -0.09108515979794671
Applying torque -0.0002519143966762834
Applying torque -0.000515783297230317
Applying torque -0.0015949139218398853
Applying torque 0.00021373268488903617
Applying torque -2.7100316825467833e-05
Applying torque -0.001

This browser does not support the video tag.


### Analysis

In [75]:
# Analyze all torques from optimization result
print("=== TORQUE ANALYSIS ===")
print(f"Analyzing torques for {len(joint_names)} joints over {num_steps} timesteps\n")

all_torques = []
torques_by_joint = {}

# Initialize joint torque storage
for joint_id, joint_name in joint_names:
    torques_by_joint[joint_name] = []

# Collect all torques
for t in range(num_steps):
    print(f"Timestep {t:2d}: ", end="")
    for joint_id, joint_name in joint_names:
        torque_key = gtd.TorqueKey(joint_id, t)
        if result.exists(torque_key):
            torque = result.atDouble(torque_key)
            all_torques.append(torque)
            torques_by_joint[joint_name].append(torque)
            print(f"{joint_name}:{torque:6.2f} ", end="")
        else:
            print(f"{joint_name}:N/A ", end="")
    print()  # New line after each timestep

print(f"\n=== OVERALL TORQUE STATISTICS ===")
print(f"Total torques collected: {len(all_torques)}")
if all_torques:
    print(f"Min torque: {min(all_torques):8.4f} Nm")
    print(f"Max torque: {max(all_torques):8.4f} Nm")
    print(f"Mean torque: {sum(all_torques)/len(all_torques):8.4f} Nm")
    print(f"Torque range: {max(all_torques) - min(all_torques):8.4f} Nm")

print(f"\n=== PER-JOINT TORQUE STATISTICS ===")
for joint_name, torques in torques_by_joint.items():
    if torques:
        print(f"{joint_name:25s}: min={min(torques):7.3f}, max={max(torques):7.3f}, mean={sum(torques)/len(torques):7.3f}, range={max(torques)-min(torques):7.3f}")
    else:
        print(f"{joint_name:25s}: No torques found")

# Find joints with highest and lowest torques
if all_torques:
    print(f"\n=== EXTREME TORQUES ===")
    # Find the joint and timestep with min/max torques
    min_torque = min(all_torques)
    max_torque = max(all_torques)
    
    for t in range(num_steps):
        for joint_id, joint_name in joint_names:
            torque_key = gtd.TorqueKey(joint_id, t)
            if result.exists(torque_key):
                torque = result.atDouble(torque_key)
                if torque == min_torque:
                    print(f"Minimum torque: {min_torque:8.4f} Nm at {joint_name} (timestep {t})")
                if torque == max_torque:
                    print(f"Maximum torque: {max_torque:8.4f} Nm at {joint_name} (timestep {t})")
else:
    print("No torques found in result!")

=== TORQUE ANALYSIS ===
Analyzing torques for 19 joints over 100 timesteps

Timestep  0: left_hip_yaw_joint:  0.02 left_hip_roll_joint: -0.00 left_hip_pitch_joint: -0.03 left_knee_joint: -0.09 left_ankle_joint: -0.00 right_hip_yaw_joint: -0.02 right_hip_roll_joint:  0.01 right_hip_pitch_joint: -0.02 right_knee_joint: -0.09 right_ankle_joint: -0.00 torso_joint: -0.00 left_shoulder_pitch_joint: -0.00 left_shoulder_roll_joint:  0.00 left_shoulder_yaw_joint: -0.00 left_elbow_joint: -0.00 right_shoulder_pitch_joint: -0.01 right_shoulder_roll_joint: -0.00 right_shoulder_yaw_joint: -0.00 right_elbow_joint: -0.00 
Timestep  1: left_hip_yaw_joint:  0.03 left_hip_roll_joint: -0.04 left_hip_pitch_joint: -0.06 left_knee_joint: -0.18 left_ankle_joint: -0.00 right_hip_yaw_joint: -0.03 right_hip_roll_joint:  0.01 right_hip_pitch_joint: -0.06 right_knee_joint: -0.18 right_ankle_joint: -0.00 torso_joint:  0.00 left_shoulder_pitch_joint: -0.01 left_shoulder_roll_joint:  0.00 left_shoulder_yaw_joint:  0.

In [74]:
# Factor Error Analysis with Plotly Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd

print("Analyzing factor errors in optimization result...")

# 1. COLLECT FACTOR ERRORS
factor_errors = []
factor_types = []
factor_descriptions = []
factor_indices = []

for idx in range(graph.size()):
    factor = graph.at(idx)
    
    try:
        # Get error for this factor
        error = factor.error(result)
        factor_errors.append(error)
        
        # Get factor type name
        factor_type = type(factor).__name__
        factor_types.append(factor_type)
        
        # Create description
        try:
            factor_keys = [str(gtd.DynamicsSymbol(key)) for key in factor.keys()]
            description = f"{factor_type}"
            if factor_keys:
                description += f" - {', '.join(factor_keys[:2])}"
                if len(factor_keys) > 2:
                    description += f" (+{len(factor_keys)-2} more)"
        except:
            description = f"{factor_type} - Factor {idx}"
        
        factor_descriptions.append(description)
        factor_indices.append(idx)
        
    except Exception as e:
        print(f"Warning: Could not analyze factor {idx}: {e}")
        factor_errors.append(0.0)
        factor_types.append("Unknown")
        factor_descriptions.append(f"Factor {idx} (Error)")
        factor_indices.append(idx)

# Create DataFrame for analysis
df_factors = pd.DataFrame({
    'Index': factor_indices,
    'Error': factor_errors,
    'Type': factor_types,
    'Description': factor_descriptions,
    'LogError': [np.log10(max(err, 1e-15)) for err in factor_errors]  # Log scale for visualization
})

# Sort by error (descending)
df_factors_sorted = df_factors.sort_values('Error', ascending=False)

print(f"Analyzed {len(factor_errors)} factors")
print(f"Total error: {sum(factor_errors):.6e}")
print(f"Max error: {max(factor_errors):.6e}")
print(f"Min error: {min(factor_errors):.6e}")

# 2. ERROR STATISTICS BY FACTOR TYPE
error_by_type = df_factors.groupby('Type').agg({
    'Error': ['count', 'sum', 'mean', 'max'],
    'Index': 'count'
}).round(8)

print("\nError statistics by factor type:")
print(error_by_type)

# 3. CREATE COMPREHENSIVE PLOTLY VISUALIZATIONS

# Create subplots with 2x2 layout
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Error Distribution by Factor Type (Bar)', 
        'Top 30 Individual Factor Errors', 
        'Error Histogram (Log Scale)', 
        'Cumulative Error Contribution'
    ),
    specs=[
        [{'type': 'bar'}, {'type': 'bar'}],
        [{'type': 'histogram'}, {'type': 'scatter'}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

# Plot 1: Total Error by Factor Type
error_by_type_sum = df_factors.groupby('Type')['Error'].sum().sort_values(ascending=False)
fig.add_trace(
    go.Bar(
        x=error_by_type_sum.index, 
        y=error_by_type_sum.values,
        name='Total Error by Type',
        text=[f'{val:.2e}' for val in error_by_type_sum.values],
        textposition='outside',
        marker_color='lightblue',
        showlegend=False
    ),
    row=1, col=1
)

# Plot 2: Top 30 Individual Factor Errors
top_30 = df_factors_sorted.head(30)
colors = ['red' if err > 1e-3 else 'orange' if err > 1e-6 else 'green' for err in top_30['Error']]
fig.add_trace(
    go.Bar(
        x=top_30['Index'], 
        y=top_30['Error'],
        text=top_30['Type'],
        textposition='outside',
        marker_color=colors,
        name='Individual Errors',
        hovertemplate='<b>Factor %{x}</b><br>' +
                     'Error: %{y:.2e}<br>' +
                     'Type: %{text}<br>' +
                     '<extra></extra>',
        showlegend=False
    ),
    row=1, col=2
)

# Plot 3: Error Histogram (Log Scale)
log_errors = [np.log10(max(err, 1e-15)) for err in factor_errors]
fig.add_trace(
    go.Histogram(
        x=log_errors,
        nbinsx=40,
        name='Log10(Error) Distribution',
        marker_color='skyblue',
        showlegend=False
    ),
    row=2, col=1
)

# Plot 4: Cumulative Error Contribution
cumsum_errors = np.cumsum(df_factors_sorted['Error'].values)
cumsum_percentage = (cumsum_errors / cumsum_errors[-1]) * 100
fig.add_trace(
    go.Scatter(
        x=list(range(len(cumsum_errors))), 
        y=cumsum_percentage,
        mode='lines',
        name='Cumulative Error %',
        line=dict(color='red', width=2),
        showlegend=False
    ),
    row=2, col=2
)

# Update layout and axes
fig.update_layout(
    height=800,
    title_text="<b>GTDynamics Factor Graph Error Analysis</b>",
    title_x=0.5,
    showlegend=False,
    font=dict(size=11)
)

# Update individual subplot axes
fig.update_xaxes(title_text="Factor Type", row=1, col=1, tickangle=45)
fig.update_yaxes(title_text="Total Error", row=1, col=1, type="log")

fig.update_xaxes(title_text="Factor Index", row=1, col=2)
fig.update_yaxes(title_text="Error", row=1, col=2, type="log")

fig.update_xaxes(title_text="Log10(Error)", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=1)

fig.update_xaxes(title_text="Factor Index (sorted by error)", row=2, col=2)
fig.update_yaxes(title_text="Cumulative Error (%)", row=2, col=2)

fig.show()

# 4. DETAILED PIE CHART OF ERROR CONTRIBUTION
fig_pie = go.Figure(data=[go.Pie(
    labels=error_by_type_sum.index,
    values=error_by_type_sum.values,
    textinfo='label+percent+value',
    texttemplate='<b>%{label}</b><br>%{percent}<br>%{value:.2e}',
    hovertemplate='<b>%{label}</b><br>' +
                 'Error: %{value:.2e}<br>' +
                 'Percentage: %{percent}<br>' +
                 '<extra></extra>'
)])

fig_pie.update_layout(
    title="<b>Error Contribution by Factor Type</b>",
    title_x=0.5,
    font=dict(size=12)
)
fig_pie.show()

# 6. SUMMARY STATISTICS
print("\n" + "="*60)
print("FACTOR ERROR ANALYSIS SUMMARY")
print("="*60)

total_factors = len(factor_errors)
high_error_factors = len([e for e in factor_errors if e > 1e-3])
medium_error_factors = len([e for e in factor_errors if 1e-6 < e <= 1e-3])
low_error_factors = len([e for e in factor_errors if e <= 1e-6])

print(f"Total factors analyzed: {total_factors}")
print(f"High error factors (>1e-3): {high_error_factors} ({high_error_factors/total_factors*100:.1f}%)")
print(f"Medium error factors (1e-6 to 1e-3): {medium_error_factors} ({medium_error_factors/total_factors*100:.1f}%)")
print(f"Low error factors (≤1e-6): {low_error_factors} ({low_error_factors/total_factors*100:.1f}%)")

print(f"\nError range: {min(factor_errors):.2e} to {max(factor_errors):.2e}")
print(f"Mean error: {np.mean(factor_errors):.6e}")
print(f"Median error: {np.median(factor_errors):.6e}")

Analyzing factor errors in optimization result...
Analyzed 15658 factors
Total error: 1.026313e-14
Max error: 9.989795e-16
Min error: 0.000000e+00

Error statistics by factor type:
                     Error                 Index
                     count  sum mean  max  count
Type                                            
ContactHeightFactor    200  0.0  0.0  0.0    200
NonlinearFactor      15415  0.0  0.0  0.0  15415
PointGoalFactor          5  0.0  0.0  0.0      5
PriorFactorDouble       38  0.0  0.0  0.0     38



FACTOR ERROR ANALYSIS SUMMARY
Total factors analyzed: 15658
High error factors (>1e-3): 0 (0.0%)
Medium error factors (1e-6 to 1e-3): 0 (0.0%)
Low error factors (≤1e-6): 15658 (100.0%)

Error range: 0.00e+00 to 9.99e-16
Mean error: 6.554558e-19
Median error: 6.418199e-26


In [59]:
# Detailed Analysis of Factor 1759 (Highest Error Factor)
factor_idx = 1759

# Get the factor
if factor_idx < graph.size():
    factor = graph.at(factor_idx)
    
    print(f"Factor Index: {factor_idx}")
    print(f"Factor Type: {type(factor).__name__}")
    
    # Get the error for this specific factor
    try:
        error = factor.error(result)
        print(f"Error Value: {error:.6e}")
    except Exception as e:
        print(f"Error calculating error: {e}")
    
    # Print detailed factor information
    print(f"\n📋 Factor Details:")
    try:
        factor.print(f"Factor {factor_idx}", gtd.GTDKeyFormatter)
    except Exception as e:
        print(f"Error printing factor details: {e}")

DETAILED ANALYSIS OF FACTOR 1759
📍 Factor Index: 1759
🏷️  Factor Type: NonlinearFactor
❌ Error Value: 2.915168e-01

📋 Factor Details:
Factor 1759  keys = { A[11]11 F[11](10)11 F[11](11)11 F[11](15)11 V[11]11 p[11]11 }
isotropic dim=6 sigma=1e-05
ExpressionFactor with measurement: [
	0;
	0;
	0;
	0;
	0;
	0
]

🔑 Variables involved in this factor:
   Number of variables: 6
   Variable 0: A[11]11

     Value: Could not extract (type object 'gtdynamics.gtdynamics.DynamicsSymbol' has no attribute 'POSE')
   Variable 1: F[11](10)11

     Value: Could not extract (type object 'gtdynamics.gtdynamics.DynamicsSymbol' has no attribute 'POSE')
   Variable 2: F[11](11)11

     Value: Could not extract (type object 'gtdynamics.gtdynamics.DynamicsSymbol' has no attribute 'POSE')
   Variable 3: F[11](15)11

     Value: Could not extract (type object 'gtdynamics.gtdynamics.DynamicsSymbol' has no attribute 'POSE')
   Variable 4: V[11]11

     Value: Could not extract (type object 'gtdynamics.gtdynamics.Dy

## Pendulum

In [7]:
import gtdynamics as gtd
import gtsam 
import numpy as np
import plotly 
import mujoco
import mujoco.viewer
import mediapy as media

#from gtsam import Point3

In [17]:
PENDULUM_PATH = "../../models/urdfs"
URDF_PATH = PENDULUM_PATH + "/inverted_pendulum.urdf"
MJCF_PATH = PENDULUM_PATH + "/scene.xml"

In [18]:
robot = gtd.CreateRobotFromFile(URDF_PATH)
model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

ValueError: XML Error: Schema violation: unrecognized element
Element 'link', line 0


In [16]:
mujoco.viewer.launch(model, data)

# Quaruped

### Setup

In [1]:
import gtdynamics as gtd
import gtsam 
import numpy as np
import plotly 
import mujoco
import mujoco.viewer
import mediapy as media

#from gtsam import Point3

In [21]:
GO2_PATH = "../../models/go2_description"
URDF_PATH = GO2_PATH + "/urdf/go2_description.urdf"
MJCF_PATH = GO2_PATH + "/mjcf/scene.xml"

In [22]:
robot = gtd.CreateRobotFromFile(URDF_PATH)
model = mujoco.MjModel.from_xml_path(MJCF_PATH)
data = mujoco.MjData(model)

In [ ]:
mujoco.viewer.launch(model, data)

In [5]:
# Noise models.
sigma_dynamics = 1e-5    # std of dynamics constraints.
sigma_objectives = 1e-6  # std of additional objectives.
sigma_joints = 1.85e-4   # 1.85e-4

dynamics_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_dynamics)
dynamics_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_dynamics)
dynamics_model_1_2 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_joints)
objectives_model_6 = gtsam.noiseModel.Isotropic.Sigma(6, sigma_objectives)
objectives_model_1 = gtsam.noiseModel.Isotropic.Sigma(1, sigma_objectives)


In [6]:
link_names = [(link.id(), link.name()) for link in robot.links()]
link_names.sort()
print(link_names)

[(0, 'base'), (1, 'FL_hip'), (2, 'FL_thigh'), (3, 'FL_calf'), (4, 'FR_hip'), (5, 'FR_thigh'), (6, 'FR_calf'), (7, 'RL_hip'), (8, 'RL_thigh'), (9, 'RL_calf'), (10, 'RR_hip'), (11, 'RR_thigh'), (12, 'RR_calf')]


In [7]:
joint_names = [(joint.id(), joint.name()) for joint in robot.joints()]
joint_names.sort()
print(joint_names)

[(0, 'FL_hip_joint'), (1, 'FL_thigh_joint'), (2, 'FL_calf_joint'), (3, 'FR_hip_joint'), (4, 'FR_thigh_joint'), (5, 'FR_calf_joint'), (6, 'RL_hip_joint'), (7, 'RL_thigh_joint'), (8, 'RL_calf_joint'), (9, 'RR_hip_joint'), (10, 'RR_thigh_joint'), (11, 'RR_calf_joint')]


In [11]:
fl_calf = robot.link("FL_calf")
fr_calf = robot.link("FR_calf")
rl_calf = robot.link("RL_calf")
rr_calf = robot.link("RR_calf")

contact_in_com = gtsam.Point3(np.array([-0.002, 0.0, -0.213]))
fl_point = gtd.PointOnLink(fl_calf, contact_in_com)
fr_point = gtd.PointOnLink(fr_calf, contact_in_com)
rl_point = gtd.PointOnLink(rl_calf, contact_in_com)
rr_point = gtd.PointOnLink(rr_calf, contact_in_com)

In [10]:
time_horizon = 5.0  # seconds
num_steps = 100
dt = time_horizon / num_steps